# FDSI benchmark v1.1: apply the winning config -- roa

The v1.1 counterpart of `../fdsi_benchmark/04c_roa_application.ipynb`, applying `03c`'s oracle
`roa` winner to all 100 recordings.

## Config

In [ ]:
RUN_APPLY_BEST = False  # compute missing adaptation_v1_1/<sub>/lr_fixed/<sampler>/*.pkl

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

DATA_DIR = Path(grid['data_root'])           # Raw archive root: <sub>/{clean,noisy}/
OUTPUTS_ROOT = Path(grid['outputs_root'])    # Precomputed-outputs archive root
CAL_DIR = OUTPUTS_ROOT / 'calibration'       # Shared with v1.0 -- written by ../fdsi_benchmark/01_calibration
ADAPT_DIR = OUTPUTS_ROOT / 'adaptation_v1_1' # v1.1 results
ADAPT_DIR_V10 = OUTPUTS_ROOT / 'adaptation'  # Cached v1.0 results, read-only here
ADAPT_DIRS = {'v1.0': ADAPT_DIR_V10, 'v1.1': ADAPT_DIR}
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')  # v1.0's base config
LR_MODE = 'fixed'   # v1.1 searches lr_fixed only -- the branch v1.0 promoted as its headline
DEVICE = 'cpu'      # deterministic, and MPS lacks the QR the sv update needs

BRANCHES = ['roa']

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')
print(f'Calibration window [0, {CAL_END}) samples; adaptation starts at sample {CAL_END}')

## Calibration

v1.1 reuses the calibrations `../fdsi_benchmark/01_calibration.ipynb` wrote, so that any v1.0 vs
v1.1 difference comes from the adaptation stage alone. Nothing is recomputed here.

In [ ]:
import sys
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, '../fdsi_benchmark')  # for fdsi_common, shared with the v1.0 benchmark
sys.path.insert(0, str(Path.cwd()))  # this notebook's own directory, for fdsi_v11

from typing import Dict, Optional, Tuple
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

import numpy as np
from adapt_decomp import CBSSResult
import fdsi_common as fc
import fdsi_v11 as fv

calibrations: Dict[Tuple[str, str, int], Optional[CBSSResult]] = {}
for sub in SUBJECTS:
    for cond in CONDITIONS:
        for snr in SNR_LEVELS:
            cal_path, _ = fc.calibration_paths(CAL_DIR, sub, cond, snr)
            calibrations[(sub, cond, snr)] = CBSSResult.load(cal_path) if cal_path.exists() else None

n_cal = sum(v is not None for v in calibrations.values())
print(f'Calibration available: {n_cal}/{len(calibrations)} recordings.')

## Read back the winning configs

In [ ]:
from adapt_decomp.adaptation import AdaptConfig

best_configs = {}
for branch in BRANCHES:
    path = fv.promoted_path(branch)
    if not path.exists():
        raise FileNotFoundError(
            f'{path.name} not found -- run 03c_*_optimisation.ipynb first to promote it.')
    best_configs[branch] = AdaptConfig.from_yaml(path)

for branch, cfg in best_configs.items():
    print(f'{branch:14} wh_lr={cfg.wh_learning_rate:.4g}  sv_lr={cfg.sv_learning_rate:.4g}  '
          f'centroid_momentum={cfg.centroid_momentum:.3g}')

## Apply the winning configs to the full benchmark grid

In [ ]:
from joblib import Parallel, delayed

from adapt_decomp import AdaptDecomp, AdaptationResult, CBSSConfig
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.spikes import get_sil


def run_adapted(sub: str, cond: str, snr: int, cbss_result: Optional[CBSSResult],
                adapt_config: AdaptConfig, sampler: Optional[str]) -> Optional[AdaptationResult]:
    """Load this recording's cached v1.1 result, or compute it if RUN_APPLY_BEST is True.

    Same steps as v1.0's run_adapted, except process_from_calib_end replaces process_data:
    the calibration window [0, CAL_END) keeps CBSS's own output and adaptation runs forwards
    from CAL_END, so the adapted segment is never scored against a window it was fitted on.

    Args:
        sub (str): Subject id.
        cond (str): Condition name.
        snr (int): SNR level in dB.
        cbss_result (Optional[CBSSResult]): This recording's calibration, or None if
            calibration itself was skipped.
        adapt_config (AdaptConfig): The config to apply.
        sampler (Optional[str]): v1.1 sampler dir, or None for the fixed baseline.

    Returns:
        Optional[AdaptationResult]: The result, or None if uncached/uncomputable.
    """
    if cbss_result is None:
        return None
    lr_mode = None if sampler is None else LR_MODE
    result_path, config_path = fv.result_paths(ADAPT_DIR, sub, cond, snr, lr_mode, sampler)
    if result_path.exists():
        return AdaptationResult.load(result_path)
    if not RUN_APPLY_BEST:
        return None

    # Read back this recording's own CBSSConfig -- ext_fact etc. must match its calibration
    emg_full = fc.load_raw_emg(DATA_DIR, sub, cond, snr)
    _, cal_config_path = fc.calibration_paths(CAL_DIR, sub, cond, snr)
    cbss_config = CBSSConfig.from_yaml(cal_config_path)

    # Initialise adaptation from calibration and adapt forwards from the calibration end
    adapter = AdaptDecomp.from_calibration(calibration=cbss_result, cbss_config=cbss_config,
                                            adapt_config=adapt_config)
    outputs = adapter.process_from_calib_end(emg_full, slice(0, CAL_END))

    # Compute SIL and RoA for the full recording
    outputs.sil = get_sil(outputs.sources, outputs.spikes, adapt_config.spike_min_dist,
                           peak_power=adapt_config.spike_det_exp).numpy()
    gt_full_bin = fc.load_gt_full_bin(DATA_DIR, sub, cond, cbss_result,
                                       n_samples=outputs.spikes.shape[0])
    outputs.roa = fc.compute_roa_for_result(outputs, gt_full_bin, FS, TOL_SPIKE_MS)

    # Save the adaptation result and config
    result_path.parent.mkdir(parents=True, exist_ok=True)
    outputs.save(result_path)
    adapt_config.to_yaml(config_path)
    return outputs


def _apply_job(sub: str, cond: str, snr: int, cal: Optional[CBSSResult],
               adapt_config: AdaptConfig, sampler: Optional[str]) -> None:
    """Run one recording in a worker process, discarding the result to keep IPC small.

    Args:
        sub (str): Subject id.
        cond (str): Condition name.
        snr (int): SNR level in dB.
        cal (Optional[CBSSResult]): This recording's calibration.
        adapt_config (AdaptConfig): The config to apply.
        sampler (Optional[str]): v1.1 sampler dir.

    Returns:
        None: The result is saved to disk by run_adapted.
    """
    run_adapted(sub, cond, snr, cal, adapt_config, sampler)


N_JOBS = 4  # Recordings run in parallel, one per worker process

jobs = [(sub, cond, snr, cal, adapt_config, fv.BRANCHES[branch].sampler)
        for branch, adapt_config in best_configs.items()
        for (sub, cond, snr), cal in calibrations.items()]
pbar = tqdm(total=len(jobs), desc='Apply best configs (v1.1)')
for _ in Parallel(n_jobs=N_JOBS, return_as='generator')(delayed(_apply_job)(*job) for job in jobs):
    pbar.update(1)
pbar.close()

print('Apply-to-all-100 pass complete (or skipped where uncached and RUN_APPLY_BEST=False).')

## Results

Everything below reads the cache off disk, so it renders on whatever `RUN_APPLY_BEST` has
already produced.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from adapt_decomp.utils.plots import plot_metric_heatmap, plot_metric_boxplot, plot_phase_bar

sns.set_theme(style='whitegrid', font_scale=1.1)

CONFIG_LABELS = fv.configs('roa')
CONFIG_ORDER = [label for label, _, _, _ in CONFIG_LABELS]

df_roa = fv.roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                      CONFIG_LABELS, FS, TOL_SPIKE_MS)
df_post = fv.window_roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                              CONFIG_LABELS, slice(CAL_END, None), FS, TOL_SPIKE_MS)
configs_present = [c for c in CONFIG_ORDER if len(df_roa) and c in set(df_roa['config'])]
print(f'Configs with cached results: {configs_present}')

### Example recording -- fixed vs adapted sources

In [ ]:
from adapt_decomp.utils.plots import plot_sources

sub, cond, snr = OPTIM_SUB, TRIANGULAR[0], OPTIM_SNR
branch = BRANCHES[0]
sampler = fv.BRANCHES[branch].sampler

fixed_outputs = AdaptationResult.load(fv.result_paths(ADAPT_DIR, sub, cond, snr, None, None)[0])
adapted_outputs = AdaptationResult.load(
    fv.result_paths(ADAPT_DIR, sub, cond, snr, LR_MODE, sampler)[0])

angle = fc.load_angle_profile(DATA_DIR, sub, cond)
ts_full = np.arange(fixed_outputs.spikes.shape[0]) / FS

axs = plot_sources(
    sources={'Adapted': adapted_outputs.sources.numpy(), 'Fixed': fixed_outputs.sources.numpy()},
    timestamps=ts_full,
    spikes={'Adapted': adapted_outputs.spikes.numpy(), 'Fixed': fixed_outputs.spikes.numpy()},
    time_range=(0, ts_full[-1]),
    square_sources=True,
)
for ax in axs:
    ax_angle = ax.twinx()
    ax_angle.plot(ts_full, angle, color='grey', alpha=0.3, zorder=0)
    ax_angle.set_yticks([])
    ax.axvline(CAL_END / FS, color='k', ls='--', lw=1)
axs[0].set_title(f'Sources -- adapted ({sampler}) vs fixed -- {sub} {cond} snr{snr}dB '
                 f'(wrist angle in grey; dashed: calibration end)')
plt.show()

### Hyperparameters -- v1.0 vs v1.1 winners

In [ ]:
fv.hyperparameter_table(BRANCHES).round(5)

### Paired per-unit RoA -- v1.0 vs v1.1

In [ ]:
if len(df_roa) and len(df_post):
    display(pd.concat({'full recording': fv.paired_summary(df_roa, fv.pairs('roa')),
                       'after calibration': fv.paired_summary(df_post, fv.pairs('roa'))}).round(3))
else:
    print('Nothing cached yet for one of the two versions.')

#### Mean / median / std RoA per adaptation approach

In [ ]:
# Both scoring windows side by side: the full recording, and only the samples after the
# calibration window, which is the one both versions actually adapted over.
WINDOW_COLS = ['version', 'config', 'condition', 'snr', 'unit', 'roa_pct']
WINDOWS = ['full recording', 'after calibration']

if len(df_roa) and len(df_post):
    df_windows = (pd.concat({'full recording': df_roa[WINDOW_COLS],
                             'after calibration': df_post[WINDOW_COLS]}, names=['window'])
                  .reset_index(level='window'))
    # Ordered categoricals so the table pairs each branch's v1.0 and v1.1 rows (the order
    # fv.configs returns) instead of grouping all of v1.0 then all of v1.1.
    df_windows['window'] = pd.Categorical(df_windows['window'], WINDOWS, ordered=True)
    df_windows['config'] = pd.Categorical(df_windows['config'], configs_present, ordered=True)
    df_windows = df_windows[df_windows['config'].notna()]

    display(df_windows.groupby(['window', 'config'], observed=True)['roa_pct']
            .agg(n_units='size', mean='mean', median='median', std='std')
            .round(2))
else:
    print('Nothing cached yet for one of the two versions.')

#### Per-unit RoA by contraction type and adaptation approach

In [ ]:
if len(df_roa) and len(df_post):
    fig, axs = plt.subplots(1, 2, figsize=(18, 5), layout='constrained', sharey=True)
    for ax, window in zip(axs, ['full recording', 'after calibration']):
        sns.boxplot(
            data=df_windows[df_windows['window'] == window],
            x='condition', y='roa_pct', hue='config',
            order=CONDITIONS, hue_order=configs_present, palette='tab10',
            flierprops=dict(marker='.', markersize=3, alpha=0.4), ax=ax,
        )
        ax.set(xlabel='Contraction type', ylabel='RoA (%)', title=f'Per-unit RoA -- {window}')
        ax.tick_params(axis='x', rotation=15)
        ax.get_legend().remove()
    axs[0].legend(title='Adaptation approach', fontsize=8, loc='lower left')
    plt.show()

### Heatmaps -- mean / median / % good units RoA

In [ ]:
if len(df_roa):
    fig, axs = plt.subplots(1, 3, figsize=(20, 4.5), layout='constrained')
    for ax, (agg, kwargs) in zip(axs, [('mean', {}), ('median', {}),
                                        ('pct_ge_threshold', {'threshold': 90})]):
        plot_metric_heatmap(df_roa, 'roa_pct', configs_present, CONDITIONS, SNR_LEVELS,
                            agg=agg, axs=[ax], **kwargs)
    plt.show()

### Per-unit RoA distribution, by condition

In [ ]:
if len(df_roa):
    fig, axs = plt.subplots(1, 1, figsize=(14, 5), layout='constrained')
    plot_metric_boxplot(df_roa, 'roa_pct', configs_present, CONDITIONS, SNR_LEVELS, axs=[axs])
    plt.show()

### Triangular-phase RoA

In [ ]:
df_phase = fv.phase_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, TRIANGULAR, SNR_LEVELS,
                          CONFIG_LABELS, FS, TOL_SPIKE_MS, CAL_END, ISO_DUR)
if len(df_phase):
    # axs defaults to a (configs x triangular conditions) grid sized by plot_phase_bar itself
    plot_phase_bar(df_phase, configs_present, TRIANGULAR, SNR_LEVELS,
                   fc.PHASE_ORDER, fc.PHASE_LABELS)
    plt.show()

### Reading this

The `after calibration` table is the one to trust for a v1.0 vs v1.1 verdict: it scores only
`[CAL_END, end)`, the window both versions actually adapted over. Pool conditions
(`POOL_CONDITIONS`) were seen by the search; `HOLDOUT_CONDITIONS` were not, so the gap between
them is the generalisation signal.